#For week 3 we are doing some data Preprocessing, utilizing the alert in the groupchat we will not using lisitng/sale dependent variables (`ListPrice`, `OriginalListPrice`,`DaysOnMarket`,`PurchaseContractDate`, and `ListingContractDate`) for model inputs. 

The final model should be able to predict closeprice for any single-family address in california (this should include homes that are not listed!)

We should filter all the data to only residential and singlefamilyresidence and we need to make sure to restrict the model population to California (Just is case any data points have a location outsdie CA)

In [ ]:
import pandas as pd
import numpy as np
import os

os.chdir(r"C:\Users\sejal\IDX\IDX-Exchange-Data-Science")
# get data
#using data from 05/2025 to 04/2026
files = [
    "Data/CRMLSSold202505.csv",
    "Data/CRMLSSold202506.csv",
    "Data/CRMLSSold202507.csv",
    "Data/CRMLSSold202508.csv",
    "Data/CRMLSSold202509.csv",
    "Data/CRMLSSold202510.csv",
    "Data/CRMLSSold202511.csv",
    "Data/CRMLSSold202512.csv",
    "Data/CRMLSSold202601.csv",
    "Data/CRMLSSold202602.csv",
    "Data/CRMLSSold202603.csv",
    "Data/CRMLSSold202604.csv"
]

#load
all_data = []

for file in files:
    temp = pd.read_csv(file, low_memory=False, dtype={"PostalCode": "string"})
    all_data.append(temp)

#combine
df = pd.concat(all_data, ignore_index=True)
#shape

print(df.shape)
#we should use errors='coerce' just in case we do see bad data, we don't want everything to crash too.
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")
print(df["CloseDate"].min())
print(df["CloseDate"].max())

# Keep California single-family homes.
sfh_df = df.loc[
    df["PropertyType"].eq("Residential")
    & df["PropertySubType"].eq("SingleFamilyResidence")
    & df["StateOrProvince"].eq("CA")
].copy()
print("California single-family rows:", len(sfh_df))


Next, we will remove exact duplicates and records without a usable price or sale date. Impossible feature values will be marked as missing. Large homes and zero bedroom or bathroom counts will stay for review.


In [ ]:
# Keep a record of what changes before splitting.
rows_before_cleaning = len(sfh_df)
exact_duplicate_rows = sfh_df.duplicated()
print("Exact duplicate rows removed:", exact_duplicate_rows.sum())
sfh_df = sfh_df.loc[~exact_duplicate_rows].copy()

numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt"
]
for col in numeric_features + ["ClosePrice"]:
    sfh_df[col] = pd.to_numeric(sfh_df[col], errors="coerce")

invalid_target = sfh_df["ClosePrice"].isna() | ~np.isfinite(sfh_df["ClosePrice"]) | (sfh_df["ClosePrice"] <= 0)
invalid_date = sfh_df["CloseDate"].isna()
print("Invalid or missing targets:", invalid_target.sum())
print("Missing sale dates:", invalid_date.sum())
sfh_df = sfh_df.loc[~(invalid_target | invalid_date)].copy()

# Impossible feature values become missing, not invented replacements.
invalid_feature_counts = {}
for col in numeric_features:
    invalid = sfh_df[col].notna() & ~np.isfinite(sfh_df[col])
    if col in ["LivingArea", "LotSizeSquareFeet"]:
        invalid |= sfh_df[col] <= 0
    elif col in ["BedroomsTotal", "BathroomsTotalInteger"]:
        invalid |= (sfh_df[col] < 0) | (sfh_df[col] % 1 != 0)
    else:
        invalid |= (sfh_df[col] <= 0) | (sfh_df[col] % 1 != 0)
        invalid |= sfh_df[col] > sfh_df["CloseDate"].dt.year
    invalid_feature_counts[col] = int(invalid.sum())
    sfh_df.loc[invalid, col] = np.nan

display(pd.Series(invalid_feature_counts, name="Values changed to missing"))

# Tidy location text without guessing missing addresses.
sfh_df["City"] = sfh_df["City"].astype("string").str.strip().str.upper().replace("", pd.NA)
postal = sfh_df["PostalCode"].astype("string").str.strip().replace("", pd.NA)
postal = postal.str.replace(r"^(\d{5})\.0$", r"\1", regex=True)
postal = postal.str.replace(r"^(\d{5})-\d{4}$", r"\1", regex=True)
sfh_df["PostalCode"] = postal
print("Rows retained:", len(sfh_df), "of", rows_before_cleaning)


EDA flagged these 15 sale prices as questionable. We will set those records aside before splitting the data and save them for review. Their correct prices are still unverified, so we will exclude them from modeling rather than guess replacements. This same list applies to all months.


In [ ]:
# Set aside the prices flagged in EDA until they can be verified.
suspect_listing_keys = [
    1133720285, 1076630163, 1136956787, 1125862013, 1115065523,
    1149697906, 1150712418, 1131758129, 1126046881, 1148201765,
    1153305774, 1116006224, 1112060979, 1135757473, 1111473680
]
suspect_price_mask = pd.to_numeric(
    sfh_df["ListingKey"], errors="coerce"
).isin(suspect_listing_keys)

suspect_price_review = sfh_df.loc[suspect_price_mask].copy()
suspect_price_review["ExclusionReason"] = "Unverified suspicious ClosePrice"
suspect_price_review.to_csv(
    os.path.join("Data", "suspect_closeprice_review.csv"), index=False
)

sfh_df = sfh_df.loc[~suspect_price_mask].copy()
print("Suspect price records excluded:", len(suspect_price_review))
print("Rows remaining:", len(sfh_df))
display(suspect_price_review[
    ["ListingKey", "CloseDate", "ClosePrice", "LivingArea", "City", "ExclusionReason"]
])


Next we will check repeated sales, very small living areas, and locations across all months. Only matching transaction copies will be removed. Other questionable records will stay in the data and be saved for review.


In [ ]:
review_cols = [
    "ListingKey", "UnparsedAddress", "CloseDate", "ClosePrice",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "City", "PostalCode",
    "CountyOrParish", "Latitude", "Longitude"
]

# Only remove repeats with matching property and sale details.
known_key = sfh_df["ListingKey"].notna()
repeated_transaction = known_key & sfh_df.duplicated(
    subset=review_cols, keep="first"
)
removed_transaction_duplicates = sfh_df.loc[repeated_transaction].copy()
sfh_df = sfh_df.loc[~repeated_transaction].copy()
print("Repeated transaction copies removed:", len(removed_transaction_duplicates))

remaining_repeats = sfh_df["ListingKey"].notna() & sfh_df.duplicated(
    subset=["ListingKey", "CloseDate"], keep=False
)
conflicting_transaction_review = sfh_df.loc[remaining_repeats, review_cols].copy()
print("Same listing and date still needing review:", len(conflicting_transaction_review))
display(conflicting_transaction_review.head(20))

# Small areas stay until checked.
small_living_area = sfh_df["LivingArea"].between(0, 200, inclusive="neither")
small_living_area_review = sfh_df.loc[
    small_living_area,
    review_cols
].copy()
print("Living areas below 200 square feet:", len(small_living_area_review))
display(small_living_area_review.sort_values("LivingArea").head(20))

location_review = sfh_df[review_cols].copy()
for col in ["Latitude", "Longitude"]:
    location_review[col] = pd.to_numeric(location_review[col], errors="coerce")

missing_coordinates = location_review[["Latitude", "Longitude"]].isna().any(axis=1)
impossible_coordinates = (
    (location_review["Latitude"].notna() & ~location_review["Latitude"].between(-90, 90))
    | (location_review["Longitude"].notna() & ~location_review["Longitude"].between(-180, 180))
)

# This is a rough screen, not a state boundary check.
outside_ca_screen = ~missing_coordinates & (
    ~location_review["Latitude"].between(32, 42.1)
    | ~location_review["Longitude"].between(-125, -114)
)
postal_text = location_review["PostalCode"].astype("string")
county_text = location_review["CountyOrParish"].astype("string").str.strip()
unusual_postal = postal_text.notna() & ~postal_text.str.fullmatch(r"\d{5}", na=False)
missing_county = county_text.isna() | county_text.eq("").fillna(False)

location_review["MissingCoordinates"] = missing_coordinates
location_review["ImpossibleCoordinates"] = impossible_coordinates
location_review["OutsideCAScreen"] = outside_ca_screen
location_review["UnusualPostalFormat"] = unusual_postal
location_review["MissingCounty"] = missing_county
location_flags = [
    "MissingCoordinates", "ImpossibleCoordinates", "OutsideCAScreen",
    "UnusualPostalFormat", "MissingCounty"
]
display(location_review[location_flags].sum().rename("Rows flagged"))
location_flag = location_review[location_flags].any(axis=1)
flagged_location_review = location_review.loc[
    location_flag
].copy()
display(flagged_location_review.head(20))

# Check all months without learning any cleaning thresholds.
month_quality = pd.DataFrame({
    "Month": sfh_df["CloseDate"].dt.to_period("M"),
    "SmallLivingArea": small_living_area,
    "RepeatedListingDate": remaining_repeats,
    "LocationFlag": location_flag
})
display(month_quality.groupby("Month").agg(
    Rows=("Month", "size"),
    SmallLivingArea=("SmallLivingArea", "sum"),
    RepeatedListingDate=("RepeatedListingDate", "sum"),
    LocationFlags=("LocationFlag", "sum")
))

lot_pairs = sfh_df["LotSizeAcres"].gt(0) & sfh_df["LotSizeSquareFeet"].gt(0)
lot_ratio = sfh_df.loc[lot_pairs, "LotSizeSquareFeet"].div(
    sfh_df.loc[lot_pairs, "LotSizeAcres"] * 43560
)
lot_unit_review = sfh_df.loc[
    lot_ratio.index[~lot_ratio.between(0.9, 1.1)],
    review_cols + ["LotSizeAcres"]
].copy()
print("Lot measurements differing by more than 10%:", len(lot_unit_review))
display(lot_unit_review.head(20))

review_tables = {
    "duplicate_transaction_review": removed_transaction_duplicates,
    "conflicting_transaction_review": conflicting_transaction_review,
    "small_living_area_review": small_living_area_review,
    "location_review": flagged_location_review,
    "lot_unit_review": lot_unit_review
}
for name, table in review_tables.items():
    table.to_csv(os.path.join("Data", name + ".csv"), index=False)


These checks use fixed rules, not statistics learned from validation or test data. The location screen only catches obvious coordinate problems; it does not verify addresses or city and ZIP matches. Review files keep the flagged rows available for checking.


In [ ]:
categorical_features = ["PostalCode", "City"]
final_features = numeric_features + categorical_features + ["LotSizeMissing"]
prohibited_features = [
    "ListPrice", "OriginalListPrice", "DaysOnMarket",
    "PurchaseContractDate", "ListingContractDate"
]


The cleaning and fixed-rule checks are done. Next we will reserve April for testing and March for validation. Missing values and categories will stay unchanged until preprocessing is fitted on each training window.


In [ ]:
sfh_df["CloseMonth"] = sfh_df["CloseDate"].dt.to_period("M")
sfh_df["LotSizeMissing"] = sfh_df["LotSizeSquareFeet"].isna().astype(int)
test_month = sfh_df["CloseMonth"].max()
validation_month = test_month - 1


In [ ]:
# Split by sale month.
test_df = sfh_df.loc[sfh_df["CloseMonth"].eq(test_month)].copy()
historical_df = sfh_df.loc[sfh_df["CloseMonth"].lt(test_month)].copy()
validation_df = historical_df.loc[
    historical_df["CloseMonth"].eq(validation_month)
].copy()
pre_validation_df = historical_df.loc[
    historical_df["CloseMonth"].lt(validation_month)
].copy()

print("Test month:", test_month, "Rows:", len(test_df))
print("Validation month:", validation_month, "Rows:", len(validation_df))
print("Rows before validation:", len(pre_validation_df))


The time split is ready. We will check feature distributions and unusual prices using only the months before March. Location flags and transaction checks were already handled above.


In [ ]:
# Review unusual values using only the months before validation.
display(pre_validation_df[numeric_features].describe(
    percentiles=[0.01, 0.5, 0.95, 0.99]
).T)

display(pd.DataFrame({
    "missing": pre_validation_df[numeric_features + categorical_features].isna().sum(),
    "missing_pct": pre_validation_df[numeric_features + categorical_features].isna().mean() * 100,
    "zero_values": pre_validation_df[numeric_features].eq(0).sum()
}))

for col in numeric_features:
    print("Largest values:", col)
    display(pre_validation_df.nlargest(10, col)[review_cols])

print("Oldest construction years:")
display(pre_validation_df.nsmallest(10, "YearBuilt")[review_cols])
print("Zero-bedroom or zero-bathroom records:")
display(pre_validation_df.loc[
    pre_validation_df["BedroomsTotal"].eq(0)
    | pre_validation_df["BathroomsTotalInteger"].eq(0), review_cols
].head(20))

for col in ["City", "PostalCode"]:
    print(col, "unique values:", pre_validation_df[col].nunique())
    display(pre_validation_df[col].value_counts(dropna=False).head(15))


price_review = pre_validation_df.copy()
price_review["PricePerSquareFoot"] = price_review["ClosePrice"].div(
    price_review["LivingArea"].where(price_review["LivingArea"] > 0)
)

price_review_cols = [
    "ListingKey", "UnparsedAddress", "CloseDate", "ClosePrice",
    "LivingArea", "PricePerSquareFoot", "BedroomsTotal",
    "BathroomsTotalInteger", "City", "PostalCode"
]
price_review_cols = [
    col for col in price_review_cols if col in price_review.columns
]

highest_price_records = price_review.nlargest(
    20, "ClosePrice"
)[price_review_cols]
highest_price_per_sqft_records = price_review.nlargest(
    20, "PricePerSquareFoot"
)[price_review_cols]

print("Highest closing prices in data before March 2026:")
display(highest_price_records)
print("Highest prices per square foot in data before March 2026:")
display(highest_price_per_sqft_records)


The review tables are ready. Flagged records remain available for source checks. Next we will prepare the candidate windows and export the approved features.


In [ ]:
#candidate training-window lengths
validation_windows = [3, 6, 9, 10]

#lets use a helper function that gives the correct rows for chosen X

def get_training_window(df, reference_month, months):
    start_month = reference_month - months

    train_df = df[
        (df["CloseMonth"] >= start_month) &
        (df["CloseMonth"] < reference_month)
    ].copy()

    return train_df
#lets test it
for x in validation_windows:
    train_x = get_training_window(
        pre_validation_df,
        validation_month,
        x
    )

    print(f"{x}-month training window:")
    print("Rows:", len(train_x))
    print("Start month:", train_x["CloseMonth"].min())
    print("End month:", train_x["CloseMonth"].max())
    print()

These windows will be compared in the modeling notebook. Medians, category rules, encoding, and any scaling must be fitted on each window's training rows only. After selection, refit them on the final training period before April.


In [ ]:
# Select approved columns once.
keep_cols = final_features + ["ClosePrice", "CloseDate", "CloseMonth"]
clean_historical_df = historical_df[keep_cols].copy()
clean_test_df = test_df[keep_cols].copy()
clean_historical_df["DataSplit"] = "historical"
clean_test_df["DataSplit"] = "test"
cleaned_df = pd.concat(
    [clean_historical_df, clean_test_df], ignore_index=True
)

# Check before saving.
assert cleaned_df.columns.tolist() == keep_cols + ["DataSplit"]
assert cleaned_df["ClosePrice"].notna().all()
assert np.isfinite(cleaned_df["ClosePrice"]).all()
assert cleaned_df["ClosePrice"].gt(0).all()
assert cleaned_df["CloseDate"].notna().all()
assert not set(prohibited_features).intersection(cleaned_df.columns)
assert cleaned_df.loc[cleaned_df["DataSplit"].eq("test"), "CloseMonth"].eq(test_month).all()
assert cleaned_df.loc[cleaned_df["DataSplit"].eq("historical"), "CloseMonth"].lt(test_month).all()

print("Final shape:", cleaned_df.shape)
print("Missing values retained:", cleaned_df.isna().sum().sum())
print(cleaned_df["DataSplit"].value_counts())
print("Date range:", cleaned_df["CloseDate"].min(), cleaned_df["CloseDate"].max())
output_path = os.path.join("Data", "cleaned_sfh_data.csv")
cleaned_df.to_csv(output_path, index=False)


## Preprocessing Summary

We combined the monthly files and kept California single-family residential properties. Exact duplicate rows and records without a valid target or sale date were removed. Impossible numeric feature values were marked as missing, and city and ZIP formatting was standardized.

The review tables use data before March to check feature ranges, unusual prices, location formats, and repeated listing keys. The 15 suspect price records from EDA were excluded using an explicit listing-key list and saved in `suspect_closeprice_review.csv`. Their prices were not corrected. Other extreme values and repeated keys remain for review. Final test results will describe the retained properties, excluding those suspect records.

March is reserved for window comparison and April for the final test. The exported `cleaned_sfh_data.csv` keeps missing values and original categories. Imputation, category grouping, encoding, and any scaling will be learned separately from each training window in the modeling notebook.

We also checked all months for matching transaction copies, small living areas, coordinate problems, missing counties, ZIP formatting, and inconsistent lot measurements. Matching copies were removed; other flagged records were saved for review and retained.
